# VQE for Quantum Chemistry

The Variational Quantum Eigensolver (VQE) finds the ground state energy of
molecular Hamiltonians. This notebook simulates the H2 molecule — the
"hydrogen atom" of quantum chemistry.

In [1]:
import numpy as np
from quantumium.algorithms.vqe import vqe
from quantumium.applications.chemistry import simulate_molecule, get_molecular_hamiltonian

## 1. The H2 Hamiltonian

In the STO-3G basis at equilibrium bond length (0.735 Å), the H2 Hamiltonian
in the qubit representation is:

H = -0.8105·II + 0.1721·ZI - 0.2257·IZ + 0.1709·ZZ + 0.0453·XX + 0.0453·YY

The exact ground state energy is **-1.137 Ha**.

In [2]:
ham, num_qubits = get_molecular_hamiltonian("H2")

# Exact ground state energy via diagonalization
eigenvalues = np.linalg.eigvalsh(ham)
exact_energy = eigenvalues[0]
print(f"H2 Hamiltonian: {ham.shape[0]}x{ham.shape[0]} matrix ({num_qubits} qubits)")
print(f"Exact ground state energy: {exact_energy:.6f} Ha")

H2 Hamiltonian: 4x4 matrix (2 qubits)
Exact ground state energy: -1.389387 Ha


## 2. Running VQE

VQE uses a parameterized quantum circuit (ansatz) and a classical optimizer
to minimize ⟨ψ(θ)|H|ψ(θ)⟩.

In [3]:
result = vqe(
    num_qubits=2,
    hamiltonian=ham,
    ansatz_depth=1,
    max_iter=100,
    num_starts=3,
)

print(f"VQE energy:   {result.optimal_energy:.6f} Ha")
print(f"Exact energy: {exact_energy:.6f} Ha")
print(f"Error:        {abs(result.optimal_energy - exact_energy):.6f} Ha")
print(f"Chemical accuracy (1.6 mHa): {'YES' if abs(result.optimal_energy - exact_energy) < 0.0016 else 'NO'}")

VQE energy:   -1.389386 Ha
Exact energy: -1.389387 Ha
Error:        0.000000 Ha
Chemical accuracy (1.6 mHa): YES


## 3. VQE Convergence

The optimizer iteratively improves the energy estimate.

In [4]:
energies = []
for depth in [1, 2, 3]:
    r = vqe(num_qubits=2, hamiltonian=ham, ansatz_depth=depth, max_iter=100, num_starts=3)
    energies.append((depth, r.optimal_energy))
    print(f"Depth {depth}: energy = {r.optimal_energy:.6f} Ha, error = {abs(r.optimal_energy - exact_energy):.6f}")

print(f"\nExact: {exact_energy:.6f} Ha")
print("Deeper ansatz → closer to exact energy (but more parameters to optimize)")

Depth 1: energy = -1.286450 Ha, error = 0.102937


Depth 2: energy = -1.342256 Ha, error = 0.047131


Depth 3: energy = -1.385899 Ha, error = 0.003488

Exact: -1.389387 Ha
Deeper ansatz → closer to exact energy (but more parameters to optimize)


## 4. Noisy VQE

On real quantum hardware, gate noise degrades the energy estimate.
Let's compare ideal vs noisy VQE.

In [5]:
from quantumium.core.circuit import QuantumCircuit
from quantumium.core.gate import Ry, CNOT
from quantumium.core.noise import NoiseModel, depolarizing

# Build a simple ansatz and compute <H> with and without noise
def compute_energy(params, noise_model=None):
    qc = QuantumCircuit(2)
    qc.apply(Ry(params[0]), 0)
    qc.apply(Ry(params[1]), 1)
    qc.apply(CNOT, 0, 1)
    qc.apply(Ry(params[2]), 0)
    qc.apply(Ry(params[3]), 1)
    if noise_model:
        rho = qc.density_matrix(noise_model=noise_model)
    else:
        sv = qc.statevector()
        rho = np.outer(sv, sv.conj())
    return np.real(np.trace(ham @ rho))

best_params = result.optimal_params
ideal_energy = compute_energy(best_params)

noise = NoiseModel(default_single_qubit_noise=depolarizing(0.01))
noise.add_gate_noise("CNOT", depolarizing(0.02))
noisy_energy = compute_energy(best_params, noise)

print(f"Ideal energy:  {ideal_energy:.6f} Ha")
print(f"Noisy energy:  {noisy_energy:.6f} Ha")
print(f"Noise penalty: {abs(noisy_energy - ideal_energy):.6f} Ha")

Ideal energy:  -0.580120 Ha
Noisy energy:  -0.589476 Ha
Noise penalty: 0.009356 Ha


## 5. Full Molecular Simulation

Quantumium includes a built-in H2 simulation pipeline.

In [6]:
mol = simulate_molecule("H2")
print(f"H2 ground state energy: {mol.ground_energy:.6f} Ha")
print(f"Exact energy:           {mol.exact_energy:.6f} Ha")
print(f"Error:                  {mol.error:.6f} Ha")
print(f"Chemical accuracy:      {mol.chemical_accuracy}")
print(f"Bond length: 0.735 Å (equilibrium)")
print(f"Basis: STO-3G")

H2 ground state energy: -1.389345 Ha
Exact energy:           -1.389387 Ha
Error:                  0.000041 Ha
Chemical accuracy:      True
Bond length: 0.735 Å (equilibrium)
Basis: STO-3G


## Next Steps

- **04_qplus_factoring.ipynb** — Factor integers with dramatically fewer qubits
- **05_noise_and_error_mitigation.ipynb** — Error mitigation techniques